# Working with ions

This notebook shows how to build, optimize, and filter conformational ensembles of **charged molecules** with `ensemblelab`. It is also a runnable check: every section ends with `assert` statements, so if a cell runs without error, that behavior works on your installation.

**You will learn how to**

1. write a charged molecule as SMILES and read its charge back from the `Ensemble`;
2. optimize an ion with MMFF (and optionally GFN2-xTB) and see the charge recorded in the provenance history;
3. remove duplicate conformers of an ion with `RMSDFilter`;
4. recognize what is *not* supported yet: salts and metal adducts.

**Requirements:** RDKit and ASE (installed with `ensemblelab`). The GFN2-xTB section needs the optional `tblite` backend and is skipped automatically if it is missing.

Two example ions are used throughout:

| Species | SMILES | Net charge |
|---|---|---|
| Perfluorooctanoate (PFOA anion, a deprotonated carboxylic acid) | `[O-]C(=O)C(F)(F)...F` | −1 |
| Propylammonium (a protonated amine) | `CCC[NH3+]` | +1 |

In [1]:
import importlib.util
import math
import warnings

import numpy as np

from ensemblelab import RMSDFilter, generate
from ensemblelab.optimizers import MMFFOptimizer

PFOA_ANION = "[O-]C(=O)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F"
PROPYLAMMONIUM = "CCC[NH3+]"

## 1. Writing charged SMILES

There is no separate `charge=` argument for generation: **the charge is part of the SMILES.** Put the charged atom in brackets with its sign, e.g. `[O-]` for a carboxylate oxygen or `[NH3+]` for an ammonium nitrogen. RDKit reads the formal charges, and `ensemblelab` sums them.

Generate a small ensemble of the PFOA anion and look at its summary. Note the `Charge` and `Multiplicity` lines, and that the generation step in the history records them too.

In [2]:
pfoa = generate(PFOA_ANION, n_confs=6)
pfoa.show(history=True, conformers=False)

Ensemble
----------------------------
SMILES: [O-]C(=O)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F
Atoms: 25
Charge: -1
Multiplicity: 1
Conformers: 6
Energy: uncomputed
Optimization: not run

Workflow History

1. generation
   Method: rdkit.ETKDGv3
   Requested: 6
   Generated: 6
   Charge: -1
   Multiplicity: 1
   Random seed: 42


The same values are available programmatically, both as properties and in the stored metadata:

In [3]:
print("ensemble.charge       :", pfoa.charge)
print("ensemble.multiplicity :", pfoa.multiplicity)
print("metadata['charge']    :", pfoa.metadata["charge"])
print("generation history    :", {k: pfoa.metadata["history"][0][k] for k in ("charge", "multiplicity")})

# Checks
assert pfoa.charge == -1
assert pfoa.multiplicity == 1
assert pfoa.metadata["charge"] == -1
assert pfoa.metadata["history"][0]["charge"] == -1
assert pfoa.metadata["history"][0]["multiplicity"] == 1

ensemble.charge       : -1
ensemble.multiplicity : 1
metadata['charge']    : -1
generation history    : {'charge': -1, 'multiplicity': 1}


### Where does the charge come from?

`Ensemble.charge` is **computed from the RDKit molecule** every time you read it; it is not a separately stored number. That means it can never disagree with the molecule. You can see which atom carries the charge:

In [4]:
charged_atoms = [
    (atom.GetIdx(), atom.GetSymbol(), atom.GetFormalCharge())
    for atom in pfoa.molecule.GetAtoms()
    if atom.GetFormalCharge() != 0
]
print("Charged atoms (index, element, formal charge):", charged_atoms)
print("Sum of formal charges:", sum(charge for _, _, charge in charged_atoms))

# Because it is computed, it is read-only:
try:
    pfoa.charge = 0
except AttributeError as error:
    print("Setting .charge is not allowed:", type(error).__name__)

# Checks
assert sum(charge for _, _, charge in charged_atoms) == pfoa.charge

Charged atoms (index, element, formal charge): [(0, 'O', -1)]
Sum of formal charges: -1
Setting .charge is not allowed: AttributeError


A **cation** works exactly the same way. Here is propylammonium, a protonated amine with charge +1. Neutral molecules report `Charge: 0`.

In [5]:
ammonium = generate(PROPYLAMMONIUM, n_confs=6)
ammonium.show(conformers=False)

neutral = generate("CC(=O)O", n_confs=1)   # acetic acid, for comparison
print("\nacetic acid charge:", neutral.charge)

# Checks
assert ammonium.charge == +1
assert ammonium.multiplicity == 1
assert neutral.charge == 0

Ensemble
----------------------------
SMILES: CCC[NH3+]
Atoms: 14
Charge: +1
Multiplicity: 1
Conformers: 6
Energy: uncomputed
Optimization: not run

acetic acid charge: 0


## 2. Optimizing an ion with MMFF

Optimizers need no extra settings for ions. MMFF assigns its atom types from the molecule's formal charges, and the optimization history entry records the charge it ran with.

In [6]:
with warnings.catch_warnings():
    # Some conformers may hit the step limit; that is reported, not an error.
    warnings.simplefilter("ignore", RuntimeWarning)
    pfoa_mmff = MMFFOptimizer().optimize(pfoa)
    ammonium_mmff = MMFFOptimizer().optimize(ammonium)

pfoa_mmff.show()
print("\nMMFF history entry charge:", pfoa_mmff.metadata["history"][-1]["charge"])

# Checks
for ensemble, charge in ((pfoa_mmff, -1), (ammonium_mmff, +1)):
    event = ensemble.metadata["history"][-1]
    assert event["method"] == "MMFF"
    assert event["charge"] == charge
    assert ensemble.charge == charge
    assert all(math.isfinite(c.energy) for c in ensemble.conformers)
    assert all(c.energy_unit == "kcal/mol" for c in ensemble.conformers)

Ensemble
----------------------------
SMILES: [O-]C(=O)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F
Atoms: 25
Charge: -1
Multiplicity: 1
Conformers: 6
Energy: computed
Optimization: MMFF

Conformers
------------------------------------
+----+------------------+--------------------+--------+-----------+-------+
| ID | Energy           | Delta E (kcal/mol) | Method | Converged | Atoms |
+----+------------------+--------------------+--------+-----------+-------+
| 0  | 124.898 kcal/mol | 0.000              | MMFF   | ✓         | 25    |
| 1  | 131.410 kcal/mol | 6.512              | MMFF   | ✓         | 25    |
| 2  | 127.056 kcal/mol | 2.158              | MMFF   | ✓         | 25    |
| 3  | 133.333 kcal/mol | 8.435              | MMFF   | ✓         | 25    |
| 4  | 126.130 kcal/mol | 1.231              | MMFF   | ✓         | 25    |
| 5  | 134.892 kcal/mol | 9.994              | MMFF   | ✓         | 25    |
+----+------------------+--------------------+--------+-----------+-------

> **Compare energies only within one charge state.** The energies above are comparable *between conformers of the PFOA anion*, so relative energies (Delta E), Boltzmann populations, and energy filters are meaningful. An anion and its neutral parent acid are different chemical species, so subtracting the acid's energy from the anion's tells you nothing about conformations. Do not compare anion energies to neutral energies.

## 3. Optimizing an ion with GFN2-xTB (optional)

Quantum-chemical backends need the charge explicitly. `GFN2xTBOptimizer` (and `ORCAOptimizer`) take it from `ensemble.charge` and `ensemble.multiplicity` by default, so you normally pass nothing. If you need a different electronic state, pass `charge=` / `multiplicity=` to the optimizer, and the history records the value actually used.

This cell runs only if `tblite` is installed. `max_steps` is kept small so the demo is quick; use the default (500) for real work.

In [7]:
if importlib.util.find_spec("tblite") is None:
    print("tblite is not installed: skipping the GFN2-xTB example.")
    print("Install it with `conda install -c conda-forge tblite-python` (macOS/Windows)")
    print("or `pip install ensemblelab[xtb]` (Linux).")
else:
    from ensemblelab.optimizers import GFN2xTBOptimizer

    for smiles, expected in ((PFOA_ANION, -1), (PROPYLAMMONIUM, +1)):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            result = GFN2xTBOptimizer(max_steps=50).optimize(generate(smiles, n_confs=1))
        event = result.metadata["history"][-1]
        print(f"{smiles:>12.12}...  GFN2-xTB ran with charge={event['charge']}, "
              f"multiplicity={event['multiplicity']}, "
              f"energy={result.conformers[0].energy:.2f} kcal/mol")

        # Checks
        assert event["method"] == "GFN2-xTB"
        assert event["charge"] == expected
        assert event["multiplicity"] == 1
        assert math.isfinite(result.conformers[0].energy)

tblite is not installed: skipping the GFN2-xTB example.
Install it with `conda install -c conda-forge tblite-python` (macOS/Windows)
or `pip install ensemblelab[xtb]` (Linux).


## 4. Removing duplicate conformers with `RMSDFilter`

Filters and analysis work on ions unchanged. `RMSDFilter` aligns conformers with RDKit's symmetry-aware `GetBestRMS`, and it treats the two carboxylate oxygens as equivalent even though the SMILES writes one as `=O` and the other as `[O-]`. Formal charges also do not change atomic masses, so mass-weighted quantities such as the radius of gyration are unaffected.

The filter keeps the lowest-energy member of each group of near-identical conformers and records which conformers it removed and why.

In [8]:
for name, ensemble in (("PFOA anion", pfoa_mmff), ("propylammonium", ammonium_mmff)):
    filtered = RMSDFilter(cutoff=0.5).apply(ensemble)
    event = filtered.metadata["history"][-1]
    print(f"{name}: kept {event['kept_conformer_ids']}, removed {event['removed_conformer_ids']}, "
          f"charge after filtering = {filtered.charge:+d}")

    # Checks: charge is preserved and every input conformer is accounted for.
    assert filtered.charge == ensemble.charge
    assert sorted(event["kept_conformer_ids"] + event["removed_conformer_ids"]) == sorted(
        ensemble.conformer_ids
    )

    # The heavy-atom variant ignores hydrogen positions (e.g. NH3+ rotation).
    heavy = RMSDFilter(cutoff=0.5, heavy_atoms_only=True).apply(ensemble)
    assert heavy.charge == ensemble.charge

PFOA anion: kept [0, 4, 2, 1, 3, 5], removed [], charge after filtering = -1
propylammonium: kept [1, 5, 0], removed [3, 2, 4], charge after filtering = +1


### Checking that coordinates stay in sync

Every `Ensemble` stores coordinates twice: in the RDKit molecule and in each conformer's ASE `Atoms`. Each step returns a new ensemble whose two copies agree and leaves its input untouched. This helper confirms that for every step above.

In [9]:
def check_sync(ensemble, tolerance=1e-6):
    # Assert RDKit and ASE coordinates agree for every conformer.
    molecule = ensemble.molecule
    symbols = [atom.GetSymbol() for atom in molecule.GetAtoms()]
    for conformer in ensemble.conformers:
        assert conformer.get_mol() is molecule
        assert conformer.atoms.get_chemical_symbols() == symbols
        rdkit_xyz = molecule.GetConformer(conformer.id).GetPositions()
        assert np.abs(rdkit_xyz - conformer.atoms.positions).max() < tolerance

steps = {
    "generated PFOA": pfoa,
    "MMFF PFOA": pfoa_mmff,
    "filtered PFOA": RMSDFilter(cutoff=0.5).apply(pfoa_mmff),
    "generated propylammonium": ammonium,
    "MMFF propylammonium": ammonium_mmff,
    "filtered propylammonium": RMSDFilter(cutoff=0.5).apply(ammonium_mmff),
}
for label, ensemble in steps.items():
    check_sync(ensemble)
    print(f"in sync: {label}")

# Each step returned a new molecule rather than modifying its input.
assert pfoa_mmff.molecule is not pfoa.molecule
assert pfoa_mmff.conformer_ids == pfoa.conformer_ids

in sync: generated PFOA
in sync: MMFF PFOA
in sync: filtered PFOA
in sync: generated propylammonium
in sync: MMFF propylammonium
in sync: filtered propylammonium


## 5. Not supported yet: salts and metal adducts

A salt or adduct such as sodium acetate, or an [M+Na]+ ion, is written in SMILES as **two separate fragments** joined by `.`: `CC(=O)O.[Na+]`. ETKDG embeds each fragment independently and places them at arbitrary distances from each other, so the resulting "conformers" would be physically meaningless. `generate()` therefore rejects any multi-fragment SMILES with a clear error:

In [10]:
for smiles in ["CC(=O)O.[Na+]", "CC(=O)[O-].[Na+]", "[Na+].[Cl-]", "CCO.O"]:
    try:
        generate(smiles, n_confs=1)
    except ValueError as error:
        print(f"{smiles!r}: rejected")
        last_error = error
    else:
        raise AssertionError(f"{smiles!r} should have been rejected")

print("\nFull message:\n", last_error)

# Checks
assert "disconnected fragments" in str(last_error)

'CC(=O)O.[Na+]': rejected
'CC(=O)[O-].[Na+]': rejected
'[Na+].[Cl-]': rejected
'CCO.O': rejected

Full message:
 SMILES 'CCO.O' contains 2 disconnected fragments. generate() supports a single connected molecule only: ETKDG places separate fragments at arbitrary distances from each other, so conformers of salts, metal adducts (e.g. [M+Na]+ written as 'CC(=O)O.[Na+]'), or complexes would be physically meaningless. Supply the charged species on its own, e.g. 'CC(=O)[O-]'.


**What to do instead:** pass the charged species on its own, e.g. `CC(=O)[O-]` for acetate.

Two related features are deliberately left for later:

- **Metal adducts** need a physically sensible placement of the metal relative to the molecule, which is a much harder problem than single-molecule conformer generation.
- **Automatic ionization.** You write the charged SMILES yourself. There is no helper that turns a neutral molecule into [M-H]- or [M+H]+, because that requires choosing *which* proton to remove or add (the protomer question).

## Summary

- Write the charge into the SMILES (`[O-]`, `[NH3+]`); read it back with `ensemble.charge` and `ensemble.multiplicity`.
- The charge is recorded at generation and in every optimization step's history, and `show()` displays it.
- MMFF, GFN2-xTB, ORCA, filters, and analysis work on ions without extra settings.
- Compare energies only among conformers of the same charged species.
- Multi-fragment SMILES (salts, metal adducts, solvates) raise `ValueError`.

See the **Ions** section of `docs/generation.md` for the reference documentation.